In [ ]:
import sys

import pandas as pd

sys.path.append("..")

from src.cleaning import charger_stations, lire_flux_v2, nettoyer_stations

stations_brutes = charger_stations("../data/raw/flux_v2.csv")
print("lignes avant nettoyage :", len(stations_brutes))

# Contrôle n° 1 du CDC : une station décrite deux fois fausserait la jointure EF-06
print("doublons exacts :", stations_brutes.duplicated().sum())
print("id en double :", stations_brutes["id"].duplicated().sum())

# Le flux ne devrait contenir que R et A (ANO-14 ne touche que l'historique)
print(stations_brutes["pop"].value_counts(dropna=False))

# Code postal lu en texte : on vérifie qu'il fait bien 5 caractères partout
print(stations_brutes["Code postal"].str.len().value_counts(dropna=False))

# errors="coerce" transforme en NaN tout ce qui n'est pas un nombre : on peut alors compter ces valeurs
latitude_num = pd.to_numeric(stations_brutes["latitude"], errors="coerce")
longitude_num = pd.to_numeric(stations_brutes["longitude"], errors="coerce")
print("latitude illisible ou vide :", latitude_num.isna().sum())
print("longitude illisible ou vide :", longitude_num.isna().sum())

In [ ]:
stations_propres = nettoyer_stations(stations_brutes)
print(stations_propres.columns.tolist())

# Créer une nouvelle table ne doit faire perdre aucune ligne
assert len(stations_propres) == len(stations_brutes)

In [ ]:
# errors="coerce" a pu créer des manquants : on les recompte (cours p. 31)
print(stations_propres[["latitude", "longitude"]].isna().sum())
print(stations_propres[["latitude", "longitude"]].describe())

# Bornes de l'Île-de-France fixées dans ANO-11
hors_bornes = stations_propres[
    (stations_propres["latitude"] < 48) | (stations_propres["latitude"] > 49.2)
    | (stations_propres["longitude"] < 1.4) | (stations_propres["longitude"] > 3.6)
]
print("stations hors bornes :", len(hors_bornes))
hors_bornes

In [ ]:

print("lignes avant nettoyage :", len(stations_brutes))
print("lignes après nettoyage :", len(stations_propres))

# Parquet (ET-05) : plus léger qu'un CSV et conserve les types du contrat D-11.
# index=False : l'index ne contient que les numéros de ligne du flux complet, inutiles ici
stations_propres.to_parquet("../data/processed/stations_idf.parquet", index=False)

# On relit le fichier pour vérifier que rien n'a été perdu à l'enregistrement
stations_relues = pd.read_parquet("../data/processed/stations_idf.parquet")
print(stations_relues.dtypes)
assert len(stations_relues) == len(stations_propres)

In [ ]:
flux_complet = lire_flux_v2("../data/raw/flux_v2.csv")

# ANO-10 : ces stations n'ont pas de département, le filtre IDF les écarte forcément
sans_departement = flux_complet[flux_complet["code_departement"].isna()]
print("ANO-10 :", len(sans_departement), "sur", len(flux_complet))

# ANO-12 : on garde ces lignes, on mesure seulement l'écart postal / administratif
prefixe_postal = stations_propres["code_postal"].str[:2]
incoherentes = stations_propres[prefixe_postal != stations_propres["departement"]]
print("ANO-12 :", len(incoherentes), "sur", len(stations_propres))
incoherentes